In [1]:
import pandas as pd
import numpy

In [11]:
#Lit le fichier csv
csv = pd.read_csv("project_dataset.csv", sep=";")
#Supprime les doublons
clean_csv = csv.drop_duplicates()
#clean_csv = csv.dropna()

#Convertie chaque colonnes dans le bon type (Int,Float,etc)
clean_csv["Date"] = pd.to_datetime(clean_csv["Date"], format="mixed")

clean_csv["Service"] = clean_csv["Service"].astype("category")
clean_csv["Gare de départ"] = clean_csv["Gare de départ"].astype("category")
clean_csv["Gare d'arrivée"] = clean_csv["Gare d'arrivée"].astype("category")

clean_csv["Durée moyenne du trajet"] = pd.to_numeric(clean_csv["Durée moyenne du trajet"].str.replace(",","."),errors="coerce")
clean_csv["Nombre de circulations prévues"] = pd.to_numeric(clean_csv["Nombre de circulations prévues"].str.replace(",", "."), errors="coerce").astype("Int64")
clean_csv["Nombre de trains annulés"] = pd.to_numeric(clean_csv["Nombre de trains annulés"].str.replace(",", "."), errors="coerce").astype("Int64")
clean_csv["Nombre de trains en retard au départ"] = pd.to_numeric(clean_csv["Nombre de trains en retard au départ"].str.replace(",", "."), errors="coerce").astype("Int64")
clean_csv["Retard moyen des trains en retard au départ"] = pd.to_numeric(clean_csv["Retard moyen des trains en retard au départ"].str.replace(",", "."), errors="coerce")
clean_csv["Retard moyen de tous les trains au départ"] = pd.to_numeric(clean_csv["Retard moyen de tous les trains au départ"].str.replace(",", "."), errors="coerce")
clean_csv["Nombre de trains en retard à l'arrivée"] = pd.to_numeric(clean_csv["Nombre de trains en retard à l'arrivée"].str.replace(",", "."), errors="coerce").astype("Int64")
clean_csv["Retard moyen des trains en retard à l'arrivée"] = pd.to_numeric(clean_csv["Retard moyen des trains en retard à l'arrivée"].str.replace(",", "."), errors="coerce")
clean_csv["Retard moyen de tous les trains à l'arrivée"] = pd.to_numeric(clean_csv["Retard moyen de tous les trains à l'arrivée"].str.replace(",", "."), errors="coerce")
clean_csv["Nombre trains en retard > 15min"] = pd.to_numeric(clean_csv["Nombre trains en retard > 15min"].str.replace(",", "."), errors="coerce").astype("Int64")
clean_csv["Retard moyen trains en retard > 15 (si liaison concurrencée par vol)"] = pd.to_numeric(clean_csv["Retard moyen trains en retard > 15 (si liaison concurrencée par vol)"].str.replace(",", "."), errors="coerce")
clean_csv["Nombre trains en retard > 30min"] = pd.to_numeric(clean_csv["Nombre trains en retard > 30min"].str.replace(",", "."), errors="coerce").astype("Int64")
clean_csv["Nombre trains en retard > 60min"] = pd.to_numeric(clean_csv["Nombre trains en retard > 60min"].str.replace(",", "."), errors="coerce").astype("Int64")

clean_csv["Prct retard pour causes externes"] = pd.to_numeric(clean_csv["Prct retard pour causes externes"].str.replace(",", "."), errors="coerce")
clean_csv["Prct retard pour cause infrastructure"] = pd.to_numeric(clean_csv["Prct retard pour cause infrastructure"].str.replace(",", "."), errors="coerce")
clean_csv["Prct retard pour cause gestion trafic"] = pd.to_numeric(clean_csv["Prct retard pour cause gestion trafic"].str.replace(",", "."), errors="coerce")
clean_csv["Prct retard pour cause matériel roulant"] = pd.to_numeric(clean_csv["Prct retard pour cause matériel roulant"].str.replace(",", "."), errors="coerce")
clean_csv["Prct retard pour cause gestion en gare et réutilisation de matériel"] = pd.to_numeric(clean_csv["Prct retard pour cause gestion en gare et réutilisation de matériel"].str.replace(",", "."), errors="coerce")
clean_csv["Prct retard pour cause prise en compte voyageurs (affluence, gestions PSH, correspondances)"] = pd.to_numeric(clean_csv["Prct retard pour cause prise en compte voyageurs (affluence, gestions PSH, correspondances)"].str.replace(",", "."), errors="coerce")

# Extrait l'année , le mois  , le jours de chaque trajets
clean_csv["year"] =  clean_csv["Date"].dt.year
clean_csv["month"] = clean_csv["Date"].dt.month
clean_csv["day"] = clean_csv["Date"].dt.day

clean_csv.drop(columns=["Commentaire annulations","Commentaire retards au départ","Commentaire retards à l'arrivée"], inplace=True)

#Interpole les colonnes (estime les données NaN a partir des données précedente et Suivante)
clean_csv["Date"] = clean_csv["Date"].interpolate()
clean_csv["Service"] = clean_csv["Service"].fillna("National")
clean_csv["Durée moyenne du trajet"] = clean_csv["Durée moyenne du trajet"].interpolate()

clean_csv["Prct retard pour causes externes"] = clean_csv["Prct retard pour causes externes"].interpolate()
clean_csv["Prct retard pour cause infrastructure"] = clean_csv["Prct retard pour cause infrastructure"].interpolate()
clean_csv["Prct retard pour cause gestion trafic"] = clean_csv["Prct retard pour cause gestion trafic"].interpolate()
clean_csv["Prct retard pour cause matériel roulant"] = clean_csv["Prct retard pour cause matériel roulant"].interpolate()
clean_csv["Prct retard pour cause gestion en gare et réutilisation de matériel"] = clean_csv["Prct retard pour cause gestion en gare et réutilisation de matériel"].interpolate()
clean_csv["Prct retard pour cause prise en compte voyageurs (affluence, gestions PSH, correspondances)"] = clean_csv["Prct retard pour cause infrastructure"].interpolate()

# Fais la Mediane pour chaque Colonnes
Columns = ["Nombre de circulations prévues",
"Nombre de trains annulés",
"Nombre de trains en retard au départ",
"Retard moyen des trains en retard au départ",
"Retard moyen de tous les trains au départ",
"Nombre de trains en retard à l'arrivée",
"Retard moyen des trains en retard à l'arrivée",
"Retard moyen de tous les trains à l'arrivée",
"Nombre trains en retard > 15min",
"Retard moyen trains en retard > 15 (si liaison concurrencée par vol)",
"Nombre trains en retard > 30min",
"Nombre trains en retard > 60min",
"Prct retard pour causes externes"]
# Regroupe les lignes qui ont la meme valeur dans les differentes colonnes , si toute une colonnes et Null on fais la mediane de toute la colonnes 
for col in Columns:
    clean_csv[col] = clean_csv.groupby(["Gare de départ", "Gare d'arrivée"])[col].transform(lambda x: x.fillna(round(x.median()) if pd.notna(x.median()) else x))
    clean_csv[col] = clean_csv[col].fillna(round(clean_csv[col].median()))

    
#clean_csv.info()
#clean_csv.isnull().sum()

    
# Ajoute les données triée dans un nouveaux fichier csv

clean_csv.to_csv("cleaned_dataset.csv")